## Logistic Regression

### Import Dependencies

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import mean_absolute_error,mean_squared_error 

In [ ]:
# Import data
players = pd.read_csv("2.3.1.training_data.csv")

# Pick one or more input features
feature_cols = ["performance", "k/d", "matchs_played"]  # can be ["performance"] or many

# Normalize feature input to list
if isinstance(feature_cols, str):
    feature_cols = [feature_cols]

# Keep only valid columns
feature_cols = [c for c in feature_cols if c in players.columns]
if len(feature_cols) == 0:
    raise ValueError("No valid feature columns found in dataset.")

# Collect rank one-hot columns (handles duplicate names with trailing spaces)
rank_cols = [c for c in players.columns if c.startswith("rank_")]
rank_groups = {}
for c in rank_cols:
    rank_groups.setdefault(c.strip(), []).append(c)

rank_flags = pd.DataFrame(
    {
        rank_name: players[cols].astype(bool).any(axis=1)
        for rank_name, cols in rank_groups.items()
    }
)

# Keep rows that have at least one rank label
valid_rows = rank_flags.any(axis=1)

# Features (X) and target (y)
X_ranks = players.loc[
    valid_rows, feature_cols
].copy()  # always DataFrame, even with 1 feature
y_ranks = (
    rank_flags.loc[valid_rows].idxmax(axis=1).str.replace("rank_", "", regex=False)
)

# Drop rows with missing feature values
mask = X_ranks.notna().all(axis=1)
X_ranks = X_ranks.loc[mask]
y_ranks = y_ranks.loc[mask]

print("Selected features:", feature_cols)
print("X shape:", X_ranks.shape)
print("y shape:", y_ranks.shape)
print("Classes:", sorted(y_ranks.unique()))
X_ranks.head()

In [ ]:
# Matplotlib-only plot: supports 1 or many features
classes = sorted(y_ranks.unique())
colors = plt.cm.tab20.colors

if len(feature_cols) == 1:
    # 1D class plot (feature vs class bands)
    feature_x = feature_cols[0]
    rng = np.random.default_rng(42)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, rank_name in enumerate(classes):
        idx = y_ranks == rank_name
        x_vals = X_ranks.loc[idx, feature_x].to_numpy()
        y_jitter = i + rng.normal(
            0, 0.06, size=x_vals.shape[0]
        )  # slight jitter for visibility
        ax.scatter(
            x_vals,
            y_jitter,
            s=35,
            alpha=0.75,
            color=colors[i % len(colors)],
            label=rank_name,
        )

    ax.set_title(f"Rank distribution by {feature_x}")
    ax.set_xlabel(feature_x)
    ax.set_ylabel("Rank")
    ax.set_yticks(range(len(classes)))
    ax.set_yticklabels(classes)
    ax.grid(True, alpha=0.3)

else:
    # 2D class plot (uses first two selected features)
    feature_x = feature_cols[0]
    feature_y = feature_cols[1]

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, rank_name in enumerate(classes):
        idx = y_ranks == rank_name
        ax.scatter(
            X_ranks.loc[idx, feature_x],
            X_ranks.loc[idx, feature_y],
            s=35,
            alpha=0.75,
            color=colors[i % len(colors)],
            label=rank_name,
        )

    ax.set_title(f"Rank distribution: {feature_x} vs {feature_y}")
    ax.set_xlabel(feature_x)
    ax.set_ylabel(feature_y)
    ax.grid(True, alpha=0.3)

ax.legend(title="Rank", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
val_regression = LogisticRegression()

In [ ]:
val_regression.fit(X_ranks, y_ranks)

In [ ]:
# Plot logistic regression predictions (works with 1, 2, or many features)
classes = np.array(sorted(y_ranks.unique()))
class_to_int = {c: i for i, c in enumerate(classes)}
y_int = np.array([class_to_int[c] for c in y_ranks])

if len(feature_cols) == 1:
    # 1D: probability curves + observed points
    fx = feature_cols[0]
    x = X_ranks[fx].to_numpy()

    pad = (x.max() - x.min()) * 0.1 if x.max() > x.min() else 1.0
    x_grid = np.linspace(x.min() - pad, x.max() + pad, 400)

    grid_df = pd.DataFrame({fx: x_grid})
    proba = val_regression.predict_proba(grid_df)

    fig, ax = plt.subplots(figsize=(9, 6))
    for i, cls in enumerate(val_regression.classes_):
        ax.plot(x_grid, proba[:, i], linewidth=2, label=f"P({cls})")

    # observed labels (jittered near 0)
    rng = np.random.default_rng(42)
    ax.scatter(
        x,
        0.02 + rng.normal(0, 0.01, size=len(x)),
        c=y_int,
        cmap="tab20",
        s=20,
        alpha=0.45,
        edgecolor="none",
        label="Observed labels",
    )

    ax.set_title("Logistic Regression Predictions (1D)")
    ax.set_xlabel(fx)
    ax.set_ylabel("Predicted probability")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

else:
    # 2D view: use first two features; fix extra features at median
    fx, fy = feature_cols[0], feature_cols[1]
    x0 = X_ranks[fx].to_numpy()
    x1 = X_ranks[fy].to_numpy()

    pad0 = (x0.max() - x0.min()) * 0.1 if x0.max() > x0.min() else 1.0
    pad1 = (x1.max() - x1.min()) * 0.1 if x1.max() > x1.min() else 1.0
    xx, yy = np.meshgrid(
        np.linspace(x0.min() - pad0, x0.max() + pad0, 250),
        np.linspace(x1.min() - pad1, x1.max() + pad1, 250),
    )

    grid_df = pd.DataFrame(np.zeros((xx.size, len(feature_cols))), columns=feature_cols)

    # Set fixed values for all features (medians), then override first two with mesh values
    for col in feature_cols:
        grid_df[col] = X_ranks[col].median()
    grid_df[fx] = xx.ravel()
    grid_df[fy] = yy.ravel()

    pred_grid = val_regression.predict(grid_df)
    pred_int = np.array([class_to_int[c] for c in pred_grid]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.contourf(
        xx,
        yy,
        pred_int,
        levels=np.arange(len(classes) + 1) - 0.5,
        alpha=0.25,
        cmap="tab20",
    )
    scatter = ax.scatter(
        x0, x1, c=y_int, cmap="tab20", s=35, edgecolor="k", linewidth=0.3
    )

    ax.set_title("Logistic Regression Predictions")
    ax.set_xlabel(fx)
    ax.set_ylabel(fy)
    ax.grid(True, alpha=0.3)

    handles, _ = scatter.legend_elements()
    ax.legend(
        handles, classes, title="Rank", bbox_to_anchor=(1.02, 1), loc="upper left"
    )
    plt.tight_layout()
    plt.show()